## 04 — RAG question answering
Embeds the question with the same model as the chunks, retrieves the top-k chunks
from the AI Search index, builds a grounded prompt and asks the LLM.

In [0]:
%pip install databricks-ai-search openai

In [0]:
dbutils.library.restartPython()

In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
%run ./00_config

In [0]:
from databricks.ai_search.client import AISearchClient

ai_cfg = config["ai_search"]

client = AISearchClient()
index = client.get_index(endpoint_name=ai_cfg["endpoint_name"], index_name=index_name)

In [0]:
def embed_question(question: str) -> list[float]:
    """Embed the question with the same model that was used for the chunks."""
    row = spark.sql(
        f"SELECT ai_query('{ai_cfg['embedding_endpoint']}', :question) AS embedding",
        args={"question": question},
    ).first()
    return list(row["embedding"])

In [0]:
RESULT_COLUMNS = ["chunk_id", "video_id", "title", "channel_title", "category_name", "chunk_text"]


def retrieve(question: str, top_k: int | None = None, filters: dict | None = None) -> list[dict]:
    """Return the top-k chunks closest to the question, with metadata and score."""
    response = index.similarity_search(
        query_vector=embed_question(question),
        columns=RESULT_COLUMNS,
        num_results=top_k or ai_cfg["top_k"],
        filters=filters,
    )
    columns = [column["name"] for column in response["manifest"]["columns"]]
    rows = response.get("result", {}).get("data_array", [])
    return [dict(zip(columns, row)) for row in rows]

In [0]:
from databricks.sdk import WorkspaceClient
from openai import OpenAI

llm_cfg = config["llm"]

workspace = WorkspaceClient()
llm_client = OpenAI(
    api_key=workspace.config.authenticate()["Authorization"].split(" ", 1)[1],
    base_url=f"{workspace.config.host}/{llm_cfg['base_path']}",
)

In [0]:
period = config["source"]["trending_period"]

SYSTEM_PROMPT_RAG = f"""You answer questions about US YouTube trending videos from {period['start']} to {period['end']}.
Use ONLY the numbered context below. Do not use any other knowledge.
If the context does not contain the answer, reply exactly: "I don't know based on the available videos."
Cite the sources you used as [1], [2] using the context numbers.
Never invent video titles, channel names or numbers."""

SYSTEM_PROMPT_NO_RAG = "You answer questions about US YouTube trending videos. Answer briefly."


def format_context(chunks: list[dict]) -> str:
    return "\n\n".join(
        f"[{number}] {chunk['title']} ({chunk['channel_title']})\n{chunk['chunk_text']}"
        for number, chunk in enumerate(chunks, start=1)
    )


def build_messages(question: str, chunks: list[dict] | None) -> list[dict]:
    if chunks is None:
        return [
            {"role": "system", "content": SYSTEM_PROMPT_NO_RAG},
            {"role": "user", "content": question},
        ]
    return [
        {"role": "system", "content": SYSTEM_PROMPT_RAG},
        {"role": "user", "content": f"Context:\n{format_context(chunks)}\n\nQuestion: {question}"},
    ]

In [0]:
def ask_llm(messages: list[dict]) -> dict:
    response = llm_client.chat.completions.create(
        model=llm_cfg["model"],
        messages=messages,
        max_tokens=llm_cfg["max_tokens"],
        temperature=llm_cfg["temperature"],
    )
    return {
        "answer": response.choices[0].message.content,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
    }


def answer_question(question: str, use_retrieval: bool = True,
                    top_k: int | None = None, filters: dict | None = None) -> dict:
    chunks = retrieve(question, top_k=top_k, filters=filters) if use_retrieval else None
    result = ask_llm(build_messages(question, chunks))

    result["question"] = question
    result["use_retrieval"] = use_retrieval
    result["sources"] = [
        {"number": n, "chunk_id": c["chunk_id"], "title": c["title"],
         "channel": c["channel_title"], "score": round(c["score"], 3)}
        for n, c in enumerate(chunks or [], start=1)
    ]
    return result